In [ ]:
# ===================================================================
# FINAL SINGLE-CELL APP (Dual OCR + Structured Answers + Single Source)
# Public Gradio share on fixed port 7863
# ===================================================================

print("STEP 1: Installing dependencies...")
!pip install -q gradio pymupdf FlagEmbedding faiss-cpu pillow pdf2image \
    opencv-python-headless nltk torch python-doctr[torch] nest_asyncio \
    replicate requests tqdm

!apt-get install -q poppler-utils > /dev/null
print("✓ All packages installed.\n")

# ------------------ Imports & setup ------------------
import os, io, re, sys, time, base64, logging, statistics, fitz
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from tqdm import tqdm

import gradio as gr
import torch, nest_asyncio, replicate, requests
from FlagEmbedding import FlagModel, FlagReranker
from doctr.models import ocr_predictor
from pdf2image import convert_from_path
from PIL import Image, ImageDraw, ImageFont

# NLTK
import nltk
try: nltk.data.find('tokenizers/punkt')
except LookupError: nltk.download('punkt', quiet=False)
try: nltk.data.find('tokenizers/punkt_tab')
except LookupError: nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize

nest_asyncio.apply()

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', force=True)
logger = logging.getLogger("dual-ocr-rag")

print("="*80)
print("INITIALIZING (Dual OCR + Structured Answers + Single Source)")
print("="*80)

# ------------------ Config ------------------
class Config:
    REPLICATE_API_TOKEN = os.getenv("REPLICATE_API_TOKEN", "YOUR_REPLICATE_API_TOKEN")
    DEEPSEEK_REPLICATE_MODEL = "lucataco/deepseek-ocr:cb3b474fbfc56b1664c8c7841550bccecbe7b74c30e45ce938ffca1180b4dff5"
    USE_DEEPSEEK = True
    IMAGE_DPI = 300
    CHUNK_SIZE = 900
    CHUNK_OVERLAP = 180
    RETRIEVAL_K = 15
    RERANK_TOP_K = 5
    USE_GPU = torch.cuda.is_available()
    STRUCTURED_KEYS = [
        "loan amount", "interest rate", "term", "underwriting fee", "closing costs",
        "principal & interest", "monthly payment", "appraisal fee", "tax service fee",
        "credit report fee", "flood certification fee", "document preparation fee",
        "notary fee", "lender's title insurance", "daily interest", "hazard insurance",
        # extras requested
        "employee id", "tax deduction", "working hours per day"
    ]

device = "cuda" if Config.USE_GPU else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if Config.USE_GPU else "None (CPU)")

# ------------------ Models ------------------
print("\n[1/4] Loading DocTR...")
p = tqdm(total=100, desc="DocTR", bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt}')
doctr_model = ocr_predictor('db_resnet50', 'crnn_vgg16_bn', pretrained=True, export_as_straight_boxes=True)
if Config.USE_GPU: doctr_model.to('cuda')
p.update(100); p.close(); print("✓ DocTR ready")

print("\n[2/4] Loading BGE + Reranker...")
p = tqdm(total=100, desc="BGE", bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt}')
embedding_model = FlagModel("BAAI/bge-large-en-v1.5", use_fp16=Config.USE_GPU)
p.update(60)
reranker_model = FlagReranker("BAAI/bge-reranker-large", use_fp16=Config.USE_GPU)
p.update(40); p.close(); print("✓ Embeddings & Reranker ready")

# ------------------ Data classes ------------------
@dataclass
class BoundingBox:
    x0: float; y0: float; x1: float; y1: float; text: str; confidence: float

@dataclass
class ExtractionResult:
    text: str; method: str; confidence: float; processing_time: float; word_count: int
    bboxes: List[BoundingBox] = field(default_factory=list)

@dataclass
class PageComparisonData:
    page_num: int
    deepseek_result: ExtractionResult
    doctr_result: ExtractionResult
    image_path: str

@dataclass
class DocumentChunk:
    text: str; doc_id: str; chunk_id: int; page_num: int; char_start: int; char_end: int
    engine: str  # "DeepSeek" or "DocTR"
    embedding: Optional[np.ndarray] = None

# ------------------ Utilities ------------------
def _clean_text(s: str) -> str:
    if not s: return ""
    s = s.replace("\u00A0", " ")  # NBSP
    s = s.replace("**", "")
    s = s.replace("\n", " ")
    s = re.sub(r"[ \t]+", " ", s)
    return s.strip()

def _format_source(page: int, engine: str, snippet: str = "") -> str:
    snip = f" | …{snippet[:80]}…" if snippet else ""
    return f"Page {page} | {engine}{snip}"

# ------------------ OCR ------------------
class DualOCREngine:
    def __init__(self):
        self.doctr_model = doctr_model
        if Config.USE_DEEPSEEK and not Config.REPLICATE_API_TOKEN:
            raise ValueError("REPLICATE_API_TOKEN not set")

    def extract_doctr(self, image: Image.Image) -> ExtractionResult:
        t0 = time.time()
        try:
            arr = np.array(image)
            res = self.doctr_model([arr]); js = res.export()
            lines, confs, bbs = [], [], []
            if not js['pages'] or not js['pages'][0]['blocks']:
                return ExtractionResult("", "DocTR", 0.0, time.time()-t0, 0)
            for block in js['pages'][0]['blocks']:
                for line in block['lines']:
                    lines.append(" ".join([w['value'] for w in line['words']]))
                    for w in line['words']:
                        confs.append(w['confidence'])
                        (x0,y0),(x1,y1) = w['geometry']
                        h,wid = arr.shape[:2]
                        bbs.append(BoundingBox(x0*wid,y0*h,x1*wid,y1*h,w['value'],w['confidence']))
            text = "\n".join(lines)
            return ExtractionResult(text,"DocTR",float(np.mean(confs)) if confs else 0.0, time.time()-t0, len(word_tokenize(text)), bbs)
        except Exception as e:
            logger.error(f"DocTR error: {e}", exc_info=True)
            return ExtractionResult("", "DocTR", 0.0, time.time()-t0, 0)

    def extract_deepseek(self, image: Image.Image) -> ExtractionResult:
        if not Config.USE_DEEPSEEK:
            return ExtractionResult("", "DeepSeek", 0.0, 0.0, 0)
        t0 = time.time()
        try:
            buf = io.BytesIO()
            try: image.save(buf, format="JPEG", quality=90); mime="image/jpeg"
            except: buf=io.BytesIO(); image.save(buf, format="PNG"); mime="image/png"
            b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
            client = replicate.Client(api_token=Config.REPLICATE_API_TOKEN)
            out = client.run(Config.DEEPSEEK_REPLICATE_MODEL, input={"image": f"data:{mime};base64,{b64}", "task_type":"Free OCR"})
            text = out if isinstance(out,str) else str(out)
            wc = len(word_tokenize(text))
            conf = 0.95 if wc>10 else 0.5
            return ExtractionResult(text,"DeepSeek",conf,time.time()-t0,wc,[])
        except Exception as e:
            logger.error(f"DeepSeek error: {e}", exc_info=True)
            return ExtractionResult("", "DeepSeek", 0.0, time.time()-t0, 0)

    def process_page(self, pdf_path: str, page_idx: int) -> PageComparisonData:
        try:
            images = convert_from_path(pdf_path, first_page=page_idx+1, last_page=page_idx+1, dpi=Config.IMAGE_DPI)
            if not images: raise RuntimeError("pdf2image returned no page image")
            img = images[0]
            d = "/tmp/ocr_images"; os.makedirs(d, exist_ok=True)
            path = os.path.join(d, f"p{page_idx+1}_{int(time.time()*1000)}.png"); img.save(path)
        except Exception as e:
            logger.error(f"Page {page_idx+1} rasterization failed: {e}", exc_info=True)
            return PageComparisonData(page_idx+1, ExtractionResult("","DeepSeek",0,0,0,[]), ExtractionResult("","DocTR",0,0,0,[]), "")

        doctr = self.extract_doctr(img)
        deep = self.extract_deepseek(img)
        logger.info(f"Page {page_idx+1} OCR: DocTR={doctr.word_count}, DeepSeek={deep.word_count}")
        return PageComparisonData(page_idx+1, deep, doctr, path)

# ------------------ Chunking ------------------
class TextChunker:
    @staticmethod
    def chunk(text: str, size: int, overlap: int)->List[Tuple[str,int,int]]:
        t = _clean_text(text)
        if not t: return []
        sents = re.split(r'(?<=[.!?])\s+', t) if re.search(r'[.!?]', t) else [t]
        chunks, cur, start = [], "", 0
        for s in sents:
            if not s: continue
            try:
                search_from = start if not chunks else max(0, chunks[-1][2]-overlap-50)
                s_start = t.index(s, search_from)
            except ValueError:
                s_start = chunks[-1][2]+1 if chunks else 0
                s_start = min(s_start, len(t)-1)
            if not cur:
                cur, start = s, s_start
            elif len(cur)+len(s)+1 <= size:
                cur += " " + s
            else:
                end = start + len(cur); chunks.append((cur,start,end))
                ov_start = max(0, end - overlap)
                cur = t[ov_start:end].strip(); start = ov_start
                if len(cur)+len(s)+1 <= size: cur += " " + s
                else:
                    chunks.append((cur,start,start+len(cur)))
                    cur, start = s, s_start
        if cur: chunks.append((cur,start,start+len(cur)))
        return chunks

# ------------------ Highlighting ------------------
class Highlight:
    @staticmethod
    def build(page: PageComparisonData, chunk: DocumentChunk)->Optional[str]:
        try:
            if not page or not page.image_path or not os.path.exists(page.image_path): return None
            bbs = page.doctr_result.bboxes  # Always DocTR boxes
            img = Image.open(page.image_path).convert("RGB")
            draw = ImageDraw.Draw(img, 'RGBA')
            if not bbs:
                draw.rectangle([10,10,img.width-10,50], fill=(255,165,0,80))
                try: font = ImageFont.load_default()
                except: font=None
                draw.text((20,20), "Highlights unavailable", fill="black", font=font)
                out = f"/tmp/highlight_stub_{int(time.time()*1000)}.png"; img.save(out); return out
            chunk_words = set(w.lower() for w in word_tokenize(chunk.text))
            hits = 0
            for b in bbs:
                words = set(w.lower() for w in word_tokenize(b.text))
                if words and not words.isdisjoint(chunk_words):
                    draw.rectangle([int(b.x0),int(b.y0),int(b.x1),int(b.y1)],
                                   fill=(255,165,0,60), outline=(255,140,0,255), width=1)
                    hits += 1
            out = f"/tmp/highlight_{int(time.time()*1000)}.png"; img.save(out)
            logger.info(f"Highlights on page {page.page_num}: {hits} boxes")
            return out
        except Exception as e:
            logger.error(f"Highlight error: {e}", exc_info=True)
            return None

# ------------------ RAG + Structured QA ------------------
import faiss

class RAG:
    def __init__(self):
        self.ocr = DualOCREngine()
        self.pages: List[PageComparisonData] = []
        self.chunks: List[DocumentChunk] = []
        self.index = None
        self.doc_id = None
        print("✓ RAG system initialized")

    def process_document(self, pdf: str, progress=gr.Progress())->Dict:
        t0 = time.time()
        doc = fitz.open(pdf); n = len(doc); doc.close()
        self.pages, self.chunks, self.index = [], [], None
        self.doc_id = f"doc_{int(time.time())}"
        total_deep, total_doc, processed = 0, 0, 0

        for i in range(n):
            progress((i+1)/(n)*0.6, desc=f"OCR {i+1}/{n}")
            pg = self.ocr.process_page(pdf, i)
            self.pages.append(pg)
            if pg.deepseek_result.word_count>0: total_deep += pg.deepseek_result.word_count
            if pg.doctr_result.word_count>0: total_doc += pg.doctr_result.word_count
            if pg.deepseek_result.word_count>0 or pg.doctr_result.word_count>0: processed += 1

        # Dual chunking
        progress(0.7, desc="Chunking")
        cid=0
        for p in self.pages:
            for (txt, eng) in [(p.deepseek_result.text, "DeepSeek"), (p.doctr_result.text, "DocTR")]:
                if not txt or not txt.strip(): continue
                for ctext, s, e in TextChunker.chunk(txt, Config.CHUNK_SIZE, Config.CHUNK_OVERLAP):
                    self.chunks.append(DocumentChunk(ctext, self.doc_id, cid, p.page_num, s, e, eng)); cid+=1
        logger.info(f"Created {len(self.chunks)} chunks.")

        # Embeddings + index
        progress(0.85, desc="Embeddings + Index")
        if self.chunks:
            texts = [c.text for c in self.chunks]
            embs = embedding_model.encode(texts)
            for c, v in zip(self.chunks, embs): c.embedding = v
            emb_arr = np.array([c.embedding for c in self.chunks]).astype('float32')
            dim = emb_arr.shape[1]
            self.index = faiss.IndexFlatL2(dim); self.index.add(emb_arr)
        progress(1.0, desc="Done")
        return {
            "success": True,
            "pages": n,
            "chunks": len(self.chunks),
            "time": time.time()-t0,
            "avg_deepseek_words": (total_deep/processed) if processed else 0,
            "avg_doctr_words": (total_doc/processed) if processed else 0,
        }

    @staticmethod
    def _doctr_bias(query: str)->float:
        q = query.lower()
        if any(k in q for k in ["amount","fee","rate","term","payment","monthly","total","insurance","interest","tax","title","employee","hours"]):
            return 0.15
        return 0.0

    def retrieve(self, query: str)->List[Tuple[DocumentChunk, float]]:
        if not (self.index and self.chunks): return []
        qv = embedding_model.encode([query])[0].astype('float32')
        D, I = self.index.search(np.array([qv]), Config.RETRIEVAL_K)
        cand = []
        for dist, idx in zip(D[0], I[0]):
            if 0<=idx<len(self.chunks):
                score = 1/(1+dist)
                cand.append((self.chunks[idx], score))
        if not cand: return []
        pairs = [[query, c.text] for c,_ in cand]
        rerank = reranker_model.compute_score(pairs, normalize=False)
        bias = self._doctr_bias(query)
        scored = []
        for (c,_), s in zip(cand, rerank):
            s = float(s)
            if c.engine == "DocTR": s += bias
            scored.append((c, s))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:Config.RERANK_TOP_K]

    # Patterns
    FIELD_PATTERNS = {
        "loan amount": re.compile(r"(?:total\s+loan\s+amount|loan\s+amount)\s*[:\-]?\s*\$?\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "interest rate": re.compile(r"(?:interest\s*rate)\s*[:\-]?\s*([0-9]+(?:\.[0-9]+)?)\s*%", re.I),
        "term": re.compile(r"(?:term|term/due\s*in)\s*[:\-]?\s*([0-9\s/]+)\s*mths?", re.I),
        "underwriting fee": re.compile(r"underwriting\s+fee.*?\$\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "closing costs": re.compile(r"(?:total\s+closing\s+costs|closing\s*/?\s*escrow\s*fee).*?\$\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "principal & interest": re.compile(r"(?:principal\s*&\s*interest|principal\s+and\s+interest)\s*\$?\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "monthly payment": re.compile(r"(?:total\s+estimated\s+monthly\s+payment|monthly\s+payment)\s*[:\-]?\s*\$?\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "appraisal fee": re.compile(r"appraisal\s+fee.*?\$\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "tax service fee": re.compile(r"tax\s+service\s+fee.*?\$\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "credit report fee": re.compile(r"credit\s+report\s+fee.*?\$\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "flood certification fee": re.compile(r"flood\s+certification\s+fee.*?\$\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "document preparation fee": re.compile(r"document\s+preparation\s+fee.*?\$\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "notary fee": re.compile(r"notary\s+fee.*?\$\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "lender's title insurance": re.compile(r"lender'?s?\s+title\s+insurance.*?\$\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "daily interest": re.compile(r"daily\s+interest.*?\$\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "hazard insurance": re.compile(r"hazard\s+insurance.*?\$\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        # new
        "employee id": re.compile(r"(?:employee\s*id)\s*[:#]?\s*([A-Za-z0-9\-]+)", re.I),
        "tax deduction": re.compile(r"(?:tax\s*(?:deduction|amount)?)[^\d]{0,10}\$?\s*([0-9][0-9,]*(?:\.\d{2})?)", re.I),
        "working hours per day": re.compile(
            r"(?:working\s*hours?(?:\s*(?:and|&)\s*rest\s*periods)?)\s*[:\-]?\s*(?:are\s*)?(?:about\s*)?"
            r"(\d+)\s*hours?(?:\s*(?:a|per)\s*day)?", re.I),
    }

    @staticmethod
    def _normalize_key(q: str)->Optional[str]:
        ql = q.lower()
        for k in Config.STRUCTURED_KEYS:
            if k in ql: return k
        # aliases
        if "employee id" in ql or ("employee" in ql and "id" in ql): return "employee id"
        if "tax" in ql and ("deduction" in ql or "amount" in ql): return "tax deduction"
        if ("working hours" in ql) or ("hours per day" in ql): return "working hours per day"
        if "rate" in ql: return "interest rate"
        if "amount" in ql and "loan" in ql: return "loan amount"
        if "term" in ql: return "term"
        return None

    def _extract_structured(self, key: str, texts_with_meta: List[Tuple[str, int, str]]):
        """
        Try: top chunks -> full text of their pages (DocTR first) -> all pages.
        Return (value, page, engine, snippet) or None.
        """
        pat = self.FIELD_PATTERNS.get(key)
        if not pat: return None

        currency_keys = {
            "loan amount","underwriting fee","closing costs","principal & interest","monthly payment",
            "appraisal fee","tax service fee","credit report fee","flood certification fee",
            "document preparation fee","notary fee","lender's title insurance","daily interest","hazard insurance",
            "tax deduction"
        }

        def format_value(k, raw):
            val = raw
            if k in currency_keys:
                val = "$" + val.replace(" ", "")
            elif k == "interest rate":
                val = val.replace(" ", "") + " %"
            elif k == "term":
                val = re.sub(r"\s+", " ", val.strip())
            return val

        def try_find(text: str):
            if not text: return None
            return pat.search(_clean_text(text))

        # (1) retrieved chunks (ranked)
        for text, page, engine in texts_with_meta:
            m = try_find(text)
            if m:
                return format_value(key, m.group(1)), page, engine, m.group(0)

        # build page list from retrieved
        retrieved_pages = []
        for _, page, _ in texts_with_meta:
            if page not in retrieved_pages: retrieved_pages.append(page)

        def try_page(page_num: int, engines=("DocTR","DeepSeek")):
            pg = next((p for p in self.pages if p.page_num == page_num), None)
            if not pg: return None
            for eng in engines:
                full = pg.doctr_result.text if eng=="DocTR" else pg.deepseek_result.text
                m = try_find(full)
                if m:
                    return format_value(key, m.group(1)), page_num, eng, m.group(0)
            return None

        # (2) full text for retrieved pages (DocTR first)
        for page in retrieved_pages:
            hit = try_page(page, ("DocTR","DeepSeek"))
            if hit: return hit

        # (3) all pages (DocTR first)
        for page in sorted({p.page_num for p in self.pages}):
            hit = try_page(page, ("DocTR","DeepSeek"))
            if hit: return hit

        return None

    # single best source selector
    def _pick_best_source(self, retrieved):
        if not retrieved: return None
        by_page = {}
        for c, score in retrieved:
            best = by_page.get(c.page_num)
            if best is None:
                by_page[c.page_num] = (c, score)
            else:
                if c.engine == "DocTR" and best[0].engine != "DocTR":
                    by_page[c.page_num] = (c, score)
        top_chunk, _ = retrieved[0]
        chosen = by_page.get(top_chunk.page_num, (top_chunk, 0.0))[0]
        return chosen, chosen.page_num, chosen.engine

    def answer(self, query: str)->Tuple[str, List[str], Optional[str]]:
        retrieved = self.retrieve(query)
        if not retrieved: return "No relevant information found.", [], None

        meta = [(c.text, c.page_num, c.engine) for c,_ in retrieved]
        key = self._normalize_key(query)

        if key:
            ext = self._extract_structured(key, meta)
            if ext:
                value, page, engine, snippet = ext
                title = key.title().replace("And", "&").replace("Lender'S","Lender's")
                answer = f"{title}: {value}"
                sources = [_format_source(page, engine, snippet)]
                page_comp = next((p for p in self.pages if p.page_num==page), None)
                chosen_chunk = next((c for c,_ in retrieved if c.page_num==page and c.engine==engine), retrieved[0][0])
                img = Highlight.build(page_comp, chosen_chunk) if page_comp else None
                return answer, sources, img

        # F1: concise fallback with SINGLE source
        chosen_chunk, page, engine = self._pick_best_source(retrieved)
        best_text = _clean_text(chosen_chunk.text)
        sents = [s.strip() for s in sent_tokenize(best_text) if s.strip()]
        if sents: fallback = " ".join(sents[:2])
        else: fallback = (best_text[:220] + "…") if len(best_text) > 220 else best_text or \
                         "Relevant information is present but could not be extracted as a single value."
        sources = [_format_source(page, engine)]
        page_comp = next((p for p in self.pages if p.page_num==page), None)
        img = Highlight.build(page_comp, chosen_chunk) if page_comp else None
        return fallback, sources, img

rag = RAG()

# ------------------ Gradio glue ------------------
def process_files(files, progress=gr.Progress()):
    if not files: return "No files uploaded", gr.update(interactive=False), None
    rag.pages, rag.chunks, rag.index = [], [], None
    total_time, total_chunks, cnt = 0.0, 0, 0
    avg_deep, avg_doc = [], []

    for f in files:
        logger.info(f"Processing: {f.name}")
        res = rag.process_document(f.name, progress)
        cnt += 1; total_time += res['time']; total_chunks += res['chunks']
        if res['avg_deepseek_words']>0: avg_deep.append(res['avg_deepseek_words'])
        if res['avg_doctr_words']>0: avg_doc.append(res['avg_doctr_words'])

    md = f"**Processed {cnt} file(s):**\n• Total Chunks: {total_chunks}\n• Total Time: {total_time:.1f}s\n\n"
    if avg_deep or avg_doc:
        md += "**OCR Comparison (Avg. Words/Page):**\n"
        if avg_deep: md += f"• DeepSeek: {statistics.mean(avg_deep):.1f}\n"
        if avg_doc: md += f"• DocTR: {statistics.mean(avg_doc):.1f}\n\n"
    md += "**Ready for queries.**" if rag.chunks else "**Warning: No text chunks extracted.**"
    return md, gr.update(interactive=bool(rag.chunks)), None

def chat_fn(message: str, history: list):
    history.append({"role":"user","content":message})
    if not rag.chunks:
        resp="Please process a valid document first."
        history.append({"role":"assistant","content":resp}); return history, None
    try:
        ans, srcs, img = rag.answer(message)
        resp = ans + ("\n\n**Source:**\n" + "\n".join(srcs) if srcs else "")
        history.append({"role":"assistant","content":resp})
        return history, img
    except Exception as e:
        logger.error(f"QA error: {e}", exc_info=True)
        resp="Sorry, an error occurred while answering."
        history.append({"role":"assistant","content":resp})
        return history, None

# ------------------ UI ------------------
custom_css = """
:root { --text-color:#eaeaea; --bg:#121212; }
.gradio-container{font-family:Inter,system-ui,sans-serif;color:var(--text-color);background:var(--bg);}
#chatbot .gr-message.user .message{background:#2a2a2a;border-radius:10px;padding:10px 15px;}
#chatbot .gr-message.bot .message{background:#1a1a1a;border:1px solid #2a2a2a;border-radius:10px;padding:10px 15px;}
.primary-btn{background:linear-gradient(135deg,#ff6b35 0%,#ff8c42 100%);color:#fff;}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Base(text_size=gr.themes.sizes.text_md)) as demo:
    with gr.Row():
        gr.Markdown("### Document Intelligence — Dual OCR + Structured Answers (Single Source)")
    with gr.Row():
        gr.Markdown("**Primary OCR:** DocTR + DeepSeek  |  **Embeddings:** BGE-large  |  **Retrieval:** FAISS + Reranker")
    with gr.Row(equal_height=False):
        with gr.Column(scale=1, min_width=320):
            gr.Markdown("#### Document Upload")
            files = gr.File(label="PDF Documents", file_count="multiple", file_types=[".pdf"])
            process_btn = gr.Button("Process Documents", elem_classes="primary-btn")
            status = gr.Markdown("**Status:** Awaiting upload")
        with gr.Column(scale=2, min_width=520):
            gr.Markdown("#### Query Interface")
            chatbot = gr.Chatbot(height=520, elem_id="chatbot", show_copy_button=True, show_label=False, type='messages')
            with gr.Row():
                msg = gr.Textbox(label=None, placeholder="Ask e.g., 'What is the loan amount?', 'Employee ID?', 'Working hours per day?'", lines=1, autofocus=True, show_label=False)
                send = gr.Button("Send", elem_classes="primary-btn")
            with gr.Row():
                examples = gr.Examples(
                    examples=[
                        "What is the loan amount?",
                        "What is the interest rate?",
                        "What is the term?",
                        "What is the underwriting fee?",
                        "What is James Bond's Employee ID?",
                        "What is the tax deduction amount?",
                        "What are the working hours per day?"
                    ],
                    inputs=msg,
                    examples_per_page=7,
                    label=""
                )
            highlighted = gr.Image(label="Source Highlight", visible=False, height=420)

    process_btn.click(process_files, inputs=files, outputs=[status, send, highlighted])
    def _clear(): return "", gr.update(visible=False)
    send.click(chat_fn, inputs=[msg, chatbot], outputs=[chatbot, highlighted])\
        .then(_clear, outputs=[msg, highlighted])\
        .then(lambda img: gr.update(visible=img is not None), inputs=highlighted, outputs=highlighted)
    msg.submit(chat_fn, inputs=[msg, chatbot], outputs=[chatbot, highlighted])\
        .then(_clear, outputs=[msg, highlighted])\
        .then(lambda img: gr.update(visible=img is not None), inputs=highlighted, outputs=highlighted)

# --- 14) LAUNCH (online public share with fixed port 7863) ---
# Make sure no offline flags are lingering
for _k in ["GRADIO_OFFLINE","GRADIO_ANALYTICS_ENABLED","GRADIO_TELEMETRY_ENABLED","GRADIO_LABS_DISABLED","GRADIO_STATS_ENABLED"]:
    os.environ.pop(_k, None)

print("\n" + "="*80); print("READY"); print("="*80)
print("Launching public interface via Gradio on port 7863...")

demo.launch(
    share=True,
    debug=False,
    server_name="0.0.0.0",
    server_port=7864,
    inline=False
)


STEP 1: Installing dependencies...
✓ All packages installed.

INITIALIZING (Dual OCR + Structured Answers + Single Source)
GPU: NVIDIA A100-SXM4-40GB

[1/4] Loading DocTR...


DocTR:   0%|          | 0/1002025-10-27 07:52:29,781 [INFO] Using downloaded & verified file: /root/.cache/doctr/models/db_resnet50-79bd7d70.pt
2025-10-27 07:52:31,858 [INFO] Using downloaded & verified file: /root/.cache/doctr/models/crnn_vgg16_bn-0417f351.pt
DocTR: 100%|██████████| 100/100


✓ DocTR ready

[2/4] Loading BGE + Reranker...


BGE: 100%|██████████| 100/100


✓ Embeddings & Reranker ready
✓ RAG system initialized

READY
Launching public interface via Gradio on port 7863...


2025-10-27 07:52:36,318 [INFO] HTTP Request: GET http://localhost:7864/gradio_api/startup-events "HTTP/1.1 200 OK"
2025-10-27 07:52:36,328 [INFO] HTTP Request: HEAD http://localhost:7864/ "HTTP/1.1 200 OK"


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()


2025-10-27 07:52:36,857 [INFO] HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2025-10-27 07:52:37,112 [INFO] HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"


* Running on public URL: https://55b9c39bfdcc663b00.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
